# アンブレラサンプリングによるDeca-alanineの自由エネルギー計算

![Graphical Abstract](./assets/figures/puling_deca_alanine.gif)

このExampleでは、典型的な生体分子モデルである **Deca-alanine (アラニン10残基)** のヘリックス-コイル転移（伸長過程）における自由エネルギープロファイルをアンブレラサンプリングで計算するワークフローを紹介します。具体的には、分子動力学シミュレーションを用いて、ペプチドの両端を引っ張りながら構造変化をサンプリングし、MBAR法を用いて自由エネルギーを算出します。

## Notebookの概要

計算は以下の6つのステップ（ノートブック）に分かれています。順番に実行することで、モデリングから解析までを一通り行うことができます。
なお、`assets`ディレクトリにそれぞれのノートブックの実行に必要なインプットを準備しているため、独立に実施することも可能です。

| Step | Notebook | 概要 |
| :--- | :--- | :--- |
| **01** | [01_modeling_peptide_ja.ipynb](./01_modeling_peptide_ja.ipynb) | **モデリング**<br>AmberToolsの `tleap` を使用して、初期構造（αヘリックス）を作成します。 |
| **02** | [02_equilibrium_nvt_md_ja.ipynb](./02_equilibrium_nvt_md_ja.ipynb) | **平衡化MD**<br>作成した構造をPFPポテンシャル下で緩和させ、安定した初期構造を得ます。 |
| **03** | [03_steered_md_ja.ipynb](./03_steered_md_ja.ipynb) | **Steered MD (SMD)**<br>PLUMEDを使用してペプチドの両端を徐々に引っ張り（または圧縮し）、反応座標全体をカバーするトラジェクトリを作成します。 |
| **04** | [04_select_umbrella_sampling_initial_structures_ja.ipynb](./04_select_umbrella_sampling_initial_structures_ja.ipynb) | **初期構造の選出**<br>SMDの軌跡から、各アンブレラサンプリング・ウィンドウ（反応座標の特定の値）に対応する構造を抽出します。 |
| **05** | [05_umbrella_sampling_ja.ipynb](./05_umbrella_sampling_ja.ipynb) | **アンブレラサンプリング**<br>抽出した構造を初期値として、調和ポテンシャルで拘束した多数のMDを並列実行し、データを収集します。 |
| **06** | [06_mbar_free_energy_ja.ipynb](./06_mbar_free_energy_ja.ipynb) | **自由エネルギー解析**<br>得られたデータをMBAR法（pymbar）で解析し、末端間距離に対する自由エネルギープロファイル（PMF）を描画します。 |

## 対象システムと反応座標

* **分子**: Deca-alanine (Ace-Ala10-Nme)
    * 両端はアセチル基(ACE)とN-メチル基(NME)でキャップされています。
    * 真空中でのシミュレーションを行います。
* **反応座標 (Collective Variable)**:
    * ペプチドの両端（N末端とC末端付近のCα原子）の距離。
    * この距離を 13.0 Å から 32.0 Å 程度まで変化させます。

## 必要なライブラリ・環境設定について

このExampleを実行するには、以下のソフトウェア・パッケージが必要です。

* **PFP-API-CLIENT**: PFP (Preferred Potential) を使用するためのパッケージ。
* **ASE (Atomic Simulation Environment)**: 原子構造の操作やMDのインターフェースとして使用。
* **AmberTools**: Step 01 での初期構造作成に使用 (`tleap` コマンド)。
* **PLUMED**: Step 03, 05 のSteered MDやUmbrella Samplingで調和振動子による束縛を加えるのに私用。
    * ASEのPLUMED Calculatorインターフェースを使用します。
* **pymbar**: Step 06 での自由エネルギー解析（MBAR法）に使用。

### PLUMEDのInstall方法について
[PLUMED](https://www.plumed.org/)は、分子動力学シミュレーションパッケージと連携して、反応座標（集団変数）の計算やバイアス力の適用を行うことで、長時間を要する稀な現象の効率的なサンプリングや自由エネルギー解析を可能にするオープンソースライブラリとなっています。本ノートブックでは、反応座標 (末端間距離)に調和振動子を追加するのにPLUMEDを使用しているため、事前にPLUMEDをインストールする必要があります。

PLUMEDのインストール手順は以下の通りとなります。

#### 1. ソースコードのダウンロードとコンパイル
ターミナルで以下のコマンドを実行し、PLUMEDをビルドします。

```bash

# インストール用のディレクトリ
mkdir -p ~/local && cd ~/local

# PLUMEDのレポジトリをクローン
git clone https://github.com/plumed/plumed2.git plumed-2.9.0

# バージョンをv2.9.0にチェックアウト
cd plumed-2.9.0 && git checkout v2.9.0

# PLUMEDのビルド (configure & make)
./configure --disable-mpi --prefix=$HOME/local/plumed-2.9.0 && make -j"$(nproc)" && make install
```

#### 2. Pythonバインディングのインストール
ASEからPLUMEDを呼び出すためのPythonパッケージもインストールします。

```bash
$ pip install plumed
```



#### ⚠️ 注意事項:

ノートブック [03_steered_md_ja.ipynb](./03_steered_md_ja.ipynb) および [05_umbrella_sampling_ja.ipynb](./05_umbrella_sampling_ja.ipynb) の冒頭には、PLUMEDのインストールパスを指定する箇所があります（例：~/local/plumed-2.9.0）。 インストールしたバージョンやディレクトリ名（plumed-2.9.0 等）に合わせて、パスを適宜修正して実行してください。